In [ ]:
import os,sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
sys.path.append('/root/liubo/TravDiT')  # 添加项目根目录到 Python 路径
from args import make_args
from util import load_raw_data,load_vec,print_metrics
import random
from tqdm import tqdm 
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
args = make_args()
unique_poi_types = args.unique_poi_types
random.seed(args.seed-1)


In [ ]:
from dataset.custom_dataset import CustomDataset
from torch.utils.data import DataLoader

all_data = []
dow_map = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
city_list = ['Changsha']
sample_size = 1000 # for one city

for city_id, city in enumerate(city_list):
    raw_data = load_raw_data(city=city)
    vec = load_vec(city=city).to(device)
    vec = vec[:, 2:2+args.poi_dim+args.pos_dim]

    indices = random.sample(range(len(raw_data)), sample_size)

    for i in indices:
        item = raw_data[i]
        item['city_id'] = 0   # ✅ 添加 city_id 字段
        vec_seq = torch.stack([vec[int(rid)] for rid in item['traj_region_id']])
        all_data.append((item, vec_seq))

# 原始数据先传入 CustomDataset
random.shuffle(all_data)
all_raw_data, all_vec_seq = zip(*all_data)
all_raw_data = list(all_raw_data)
all_vec_seq = list(all_vec_seq)

full_dataset = CustomDataset(all_raw_data, all_vec_seq)
val_dataloader = DataLoader(full_dataset, batch_size=args.Encoder_batch_size, shuffle=False)

In [ ]:
# 预训练Transformer encoder for DiT
from model.model_Encoder import ST_Encoder,ST_Decoder
from model.st_layers_config.args import parse_args

encoder_config = parse_args()
encoder = ST_Encoder(config = encoder_config,dim_in = args.poi_dim+args.pos_dim, dim_out = args.latent_dim)
decoder = ST_Decoder(latent_dim = args.latent_dim, vocab_size = args.vocab_size,city_emb_dim=16)
encoder.to(device)
decoder.to(device)
encoder.load_state_dict(torch.load(f"/root/liubo/TravDiT/{args.checkpt_path}/pretrain/encoder/encoder_3.pth", map_location=device))
decoder.load_state_dict(torch.load(f"/root/liubo/TravDiT/{args.checkpt_path}/pretrain/encoder/decoder_3.pth", map_location=device))

In [ ]:
criterion = torch.nn.CrossEntropyLoss()

with torch.no_grad():
    encoder.eval()
    decoder.eval()
    epoch_loss = 0
    y_true_list = []
    y_pred_list = []

    dataloader_tqdm = tqdm(val_dataloader, leave=False)
    for batch in dataloader_tqdm:
        region_id = batch['region_seq'].to(device)  # [B, K]
        labels = batch['region_seq'].to(device)        # [B, K]
        city_id = batch['city_id_seq'].to(device)      # [B, ]
        vec = batch['vec_seq'].to(device)  # [B, K, poi_dim + pos_dim]

        labels = labels.reshape(-1, args.K).long()  # [B*K]
        city_id = city_id.unsqueeze(1).expand(-1, args.K)  # [B, K]

        latent = encoder(input=vec.permute(0, 2, 1).unsqueeze(2))  # -> [B, T, 1, C]
        output = decoder(latent,city_id)           
        
        val_loss = criterion(output.view(-1, output.size(-1)), labels.view(-1))
        y_true = labels.view(-1).cpu()
        y_pred = output.argmax(dim=-1).view(-1).cpu()

        y_true_list.append(y_true)
        y_pred_list.append(y_pred)

    print_metrics(y_true_list, y_pred_list)
